In [ ]:
# --- Imports ---
import os
import numpy as np
import pandas as pd
import joblib # For saving the pipeline
from collections import defaultdict

# Plotting & Visualization (Optional for core logic, but good for EDA)
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
# %matplotlib inline # Keep if in Jupyter/IPython

# Scikit-learn components
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.manifold import TSNE  # For visualization
from sklearn.decomposition import PCA # For visualization
from sklearn.metrics import euclidean_distances
from scipy.spatial.distance import cdist # For calculating distances

# Optional: Spotify API (Commented out if only using local data for recommendations)
# %pip install spotipy
# import spotipy
# from spotipy.oauth2 import SpotifyClientCredentials

import warnings
warnings.filterwarnings("ignore")

# --- Configuration ---
# Define Client ID and Secret if using Spotify API (keep commented out if not)
# SPOTIPY_CLIENT_ID = 'YOUR_CLIENT_ID'  # Replace with your actual ID
# SPOTIPY_CLIENT_SECRET = 'YOUR_SECRET' # Replace with your actual Secret

# --- Data Loading ---
# Use appropriate encoding found previously (e.g., 'latin1', 'utf-8' with errors='replace')
ENCODING_METHOD = 'utf-8' # Or 'latin1', 'cp1252', etc.
ERROR_HANDLING = 'replace' # Or 'ignore', or remove if encoding is correct

try:
    # Attempt to load data
    data = pd.read_csv("data.csv", encoding=ENCODING_METHOD, errors=ERROR_HANDLING)
    genre_data = pd.read_csv('data_by_genres.csv', encoding=ENCODING_METHOD, errors=ERROR_HANDLING)
    year_data = pd.read_csv('data_by_year.csv', encoding=ENCODING_METHOD, errors=ERROR_HANDLING)
    print("Data loaded successfully.")
except FileNotFoundError:
    # If any file is not found...
    print("Error: One or more CSV files not found. Make sure they are in the correct directory.")
    exit() # <-- Script stops here
except Exception as e:
    # If any other error occurs during loading...
    print(f"Error loading data: {e}")
    exit() # <-- Script stops here

print("\n--- Data Info ---")
print("data.csv info:")
data.info() # <--- ERROR HAPPENS HERE
# print("\ndata_by_genres.csv info:") # Optional: uncomment to see genre info
# genre_data.info()
# print("\ndata_by_year.csv info:") # Optional: uncomment to see year info
# year_data.info()

# --- Feature Selection for Song Recommendation ---
# Select only numeric columns from the main song data ('data.csv')
# These columns will be used for scaling, potential clustering, and similarity calculation.
try:
    numeric_features = data.select_dtypes(include=np.number)
    # Define the list of columns ONCE based on the selection
    number_cols = list(numeric_features.columns)
    print(f"\nUsing {len(number_cols)} numeric features for recommendation:")
    print(number_cols)

    # Optional: Data Cleaning / Preprocessing for selected columns
    # Check for missing values in the selected numeric columns
    if numeric_features.isnull().sum().sum() > 0:
        print("\nWarning: Missing values found in numeric features. Filling with median.")
        # Create a copy to avoid modifying the original DataFrame directly during fillna
        X = numeric_features.copy()
        # Fill missing values (using median is often robust)
        X = X.fillna(X.median())
    else:
        X = numeric_features # No missing values, use directly

    # Ensure X is not empty
    if X.empty:
        print("Error: No numeric data found after selection/cleaning. Cannot proceed.")
        exit()

except Exception as e:
    print(f"Error during feature selection/preprocessing: {e}")
    exit()


# --- Song Clustering Pipeline Definition & Training ---
# This pipeline includes scaling and KMeans.
# The recommendation function below primarily uses the SCALER.
# The KMeans part assigns cluster labels, useful for analysis or alternative recommendation methods.
N_CLUSTERS_SONGS = 50 # Example: Choose number of song clusters (tune if needed)

song_cluster_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=N_CLUSTERS_SONGS,
                      init='k-means++',
                      n_init=10, # Run multiple times
                      random_state=42,
                      verbose=False)) # Set to True for more KMeans output
], verbose=False) # Set to True to see pipeline steps during fit

print(f"\nTraining song clustering pipeline with {N_CLUSTERS_SONGS} clusters...")
# Fit the pipeline ONLY on the selected numeric features (X)
try:
    song_cluster_pipeline.fit(X)
    print("Song clustering pipeline training complete.")

    # Add cluster labels back to the main dataframe for analysis/visualization
    song_cluster_labels = song_cluster_pipeline.predict(X)
    data['cluster_label'] = song_cluster_labels
    print("Cluster labels added to the 'data' DataFrame.")

except Exception as e:
    print(f"Error training song clustering pipeline: {e}")
    # Decide how to handle - maybe exit, maybe try to continue without clustering
    # For now, we'll exit if the main pipeline fails
    exit()


# --- Visualization (Optional) ---
# 1. Genre Clustering Visualization (using t-SNE) - Requires genre_data
print("\nPerforming t-SNE on genre data for visualization...")
try:
    # Ensure genre_data has numeric types for clustering/t-SNE
    genre_numeric = genre_data.select_dtypes(include=[np.number])
    if not genre_numeric.empty:
        # Genre Clustering (temporary for visualization)
        genre_cluster_pipeline = make_pipeline(StandardScaler(), KMeans(n_clusters=10, n_init=10, random_state=42))
        genre_data['cluster'] = genre_cluster_pipeline.fit_predict(genre_numeric)

        # t-SNE pipeline
        tsne_pipeline = make_pipeline(StandardScaler(), TSNE(n_components=2, verbose=0, random_state=42)) # verbose=0 for less output
        projection_genre = pd.DataFrame(tsne_pipeline.fit_transform(genre_numeric), columns=['x', 'y'])
        projection_genre['genres'] = genre_data['genres']
        projection_genre['cluster'] = genre_data['cluster']

        # Plot genre clusters
        fig_genre = px.scatter(projection_genre, x='x', y='y', color='cluster', hover_data=['genres'], title="Genre Clusters (t-SNE projection)")
        # fig_genre.show() # Uncomment to display the plot
        print("Genre t-SNE visualization prepared (uncomment fig_genre.show() to display).")
    else:
        print("Skipping genre visualization - no numeric columns found in genre_data.")
except Exception as e:
    print(f"Error during genre visualization: {e}")


# 2. Song Clustering Visualization (using PCA) - Requires 'data' and 'X'
print("\nPerforming PCA on song data for visualization...")
try:
    pca_pipeline = Pipeline([('scaler', StandardScaler()), ('PCA', PCA(n_components=2, random_state=42))])
    song_embedding = pca_pipeline.fit_transform(X) # Use the same numeric data X
    projection_song = pd.DataFrame(columns=['x', 'y'], data=song_embedding)
    projection_song['title'] = data['name'] # Assuming 'name' column exists
    projection_song['cluster'] = data['cluster_label'] # Use labels from the main pipeline

    # Plot song clusters
    fig_song = px.scatter(
        projection_song, x='x', y='y', color='cluster', hover_data=['x', 'y', 'title'], title="Song Clusters (PCA projection)"
    )
    # fig_song.show() # Uncomment to display the plot
    print("Song PCA visualization prepared (uncomment fig_song.show() to display).")
except KeyError as e:
     print(f"Skipping song visualization - Column missing: {e}. Ensure 'name' column exists.")
except Exception as e:
    print(f"Error during song visualization: {e}")


# --- Spotify API Interaction (Optional - Commented Out) ---
# Use this section ONLY if you need to fetch data for songs NOT in your CSV
# print("\nSetting up Spotify API connection...")
# try:
#     sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(client_id=SPOTIPY_CLIENT_ID,
#                                                            client_secret=SPOTIPY_CLIENT_SECRET))
#     print("Spotify connection successful.")
# except Exception as e:
#     print(f"Could not connect to Spotify API: {e}")
#     sp = None # Ensure sp is None if connection fails

# def find_song_spotify(name, year, sp_client):
#     """Fetches song data from Spotify API."""
#     if sp_client is None:
#         print("Spotify client not available.")
#         return None
#     song_data = defaultdict(list) # Use list for multi-value potential later
#     try:
#         query = f'track:{name} year:{year}'
#         results = sp_client.search(q=query, limit=1, type='track')

#         if not results['tracks']['items']:
#             print(f"Song '{name}' ({year}) not found on Spotify.")
#             return None

#         track_info = results['tracks']['items'][0]
#         track_id = track_info['id']
#         audio_features = sp_client.audio_features(track_id)[0]

#         # Basic Info
#         song_data['name'] = [name] # Keep as list for consistency
#         song_data['year'] = [int(year)]
#         song_data['explicit'] = [int(track_info['explicit'])]
#         song_data['duration_ms'] = [track_info['duration_ms']]
#         song_data['popularity'] = [track_info['popularity']]
#         song_data['artists'] = [', '.join([artist['name'] for artist in track_info['artists']])] # Store artists

#         # Audio Features - Ensure keys match number_cols if possible
#         for key, value in audio_features.items():
#             if key in number_cols: # Only add if it's a feature we use
#                 song_data[key] = [value] # Store as list

#         # Check for missing features we need
#         for col in number_cols:
#             if col not in song_data:
#                  # Decide how to handle missing features (e.g., fill with 0 or None)
#                  # For now, let's skip adding them if missing from Spotify's features
#                  pass

#         return pd.DataFrame(dict(song_data)) # Convert defaultdict back to dict for DataFrame

#     except Exception as e:
#         print(f"Error searching Spotify for '{name}' ({year}): {e}")
#         return None


# --- Recommendation Functions ---

def get_song_data(song, spotify_data):
    """
    Retrieve song data strictly from the loaded dataset ('spotify_data')
    based on name and year.
    """
    try:
        # Normalize case for better matching (optional but recommended)
        song_name_lower = song['name'].lower()
        # Ensure year is integer for matching
        song_year = int(song['year'])

        # Filter using normalized name and integer year
        song_matches = spotify_data[
            (spotify_data['name'].str.lower() == song_name_lower) &
            (spotify_data['year'] == song_year)
        ]

        if song_matches.empty:
             print(f"Warning: Song '{song['name']}' ({song['year']}) not found in the local dataset.")
             return None
        # Return the first match if multiple exist
        return song_matches.iloc[0]

    except KeyError:
        print(f"Error: Input song dictionary must have 'name' and 'year' keys. Got: {song}")
        return None
    except Exception as e:
        print(f"Error retrieving local song data for '{song.get('name', 'N/A')}': {e}")
        return None


def get_mean_vector(song_list, spotify_data, feature_cols):
    """
    Compute the average feature vector for a list of songs based on the local dataset.
    Uses the provided list of feature_cols.
    """
    song_vectors = []

    for song_input in song_list:
        song_data = get_song_data(song_input, spotify_data)
        if song_data is None:
            # Warning already printed in get_song_data
            continue # Skip this song

        # Ensure all required feature columns are present in the found song_data
        if not all(col in song_data for col in feature_cols):
            print(f"Warning: Missing some numeric features for '{song_input['name']}'. Skipping this song.")
            continue

        # Extract the vector for the specified feature columns
        try:
            song_vector = song_data[feature_cols].values.astype(np.float64) # Ensure float type
            song_vectors.append(song_vector)
        except Exception as e:
            print(f"Error extracting vector for '{song_input['name']}': {e}")
            continue # Skip if vector extraction fails

    if not song_vectors: # Check if the list is empty
        print("Error: No valid song vectors found for the input list. Cannot compute mean vector.")
        return None # Indicate failure

    # Compute the mean vector
    song_matrix = np.array(song_vectors)
    mean_vector = np.mean(song_matrix, axis=0)
    return mean_vector


def flatten_dict_list(dict_list):
    """Flatten a list of dictionaries into a dictionary of lists."""
    flattened_dict = defaultdict(list)
    for dictionary in dict_list:
        for key, value in dictionary.items():
            flattened_dict[key].append(value)
    return flattened_dict


def recommend_songs(song_list, spotify_data, pipeline, feature_cols, n_songs=10):
    """
    Recommend songs based on cosine similarity to the mean vector of the input song_list.
    Uses the scaler from the provided pipeline and operates on the specified feature_cols.
    """
    metadata_cols = ['name', 'year', 'artists'] # Columns to return for recommended songs

    # --- Input Validation ---
    if not isinstance(song_list, list) or not song_list:
        print("Error: Input 'song_list' must be a non-empty list of dictionaries.")
        return pd.DataFrame(columns=metadata_cols) # Return empty DataFrame

    if spotify_data.empty:
        print("Error: The 'spotify_data' DataFrame is empty.")
        return pd.DataFrame(columns=metadata_cols)

    if pipeline is None or not hasattr(pipeline, 'steps') or len(pipeline.steps) < 1:
        print("Error: Invalid or untrained 'pipeline' provided.")
        return pd.DataFrame(columns=metadata_cols)

    if not feature_cols or not all(col in spotify_data.columns for col in feature_cols):
         print(f"Error: Not all feature_cols {feature_cols} found in spotify_data columns.")
         return pd.DataFrame(columns=metadata_cols)

    # --- Feature Vector Calculation ---
    song_dict = flatten_dict_list(song_list) # For filtering later
    song_center = get_mean_vector(song_list, spotify_data, feature_cols)

    if song_center is None:
        print("Error: Could not compute mean vector for input songs. Aborting recommendation.")
        return pd.DataFrame(columns=metadata_cols) # Return empty DataFrame

    # --- Scaling and Distance Calculation ---
    try:
        # Extract the scaler (should be the first step)
        scaler = pipeline.steps[0][1]
        if not isinstance(scaler, StandardScaler): # Basic check
             print("Error: First step in pipeline is not a StandardScaler.")
             return pd.DataFrame(columns=metadata_cols)

        # Prepare the dataset's numeric features (handle potential missing values again just in case)
        dataset_features = spotify_data[feature_cols].fillna(spotify_data[feature_cols].median())

        # Scale the dataset and the input song center
        scaled_data = scaler.transform(dataset_features)
        scaled_song_center = scaler.transform(song_center.reshape(1, -1))

        # Compute cosine distances (1 - cosine_similarity)
        # cdist calculates distance; lower is better (more similar)
        distances = cdist(scaled_song_center, scaled_data, 'cosine').flatten() # Use flatten()

    except Exception as e:
        print(f"Error during scaling or distance calculation: {e}")
        return pd.DataFrame(columns=metadata_cols)

    # --- Selecting Recommendations ---
    # Get indices of songs sorted by distance (ascending order - closest first)
    # Exclude the first index if it's trivially zero (the exact mean vector match, unlikely)
    # We want n_songs *plus* potentially the input songs to filter them out later
    # Let's get more indices than needed initially
    num_to_retrieve = n_songs + len(song_list)
    sorted_indices = np.argsort(distances)

    # Filter out the input songs from the results
    rec_songs_df = spotify_data.iloc[sorted_indices].copy() # Work with a copy
    input_song_names_lower = {s.lower() for s in song_dict['name']}
    input_song_years = {int(y) for y in song_dict['year']} # Ensure comparison with integers

    # Filter out songs that are in the input list (match both name and year)
    is_input_song = rec_songs_df.apply(
        lambda row: row['name'].lower() in input_song_names_lower and row['year'] in input_song_years,
        axis=1
    )
    rec_songs_filtered = rec_songs_df[~is_input_song]

    # Select the top N songs from the filtered list
    final_recs = rec_songs_filtered.head(n_songs)

    # --- Return Results ---
    # Check if desired metadata columns exist
    valid_metadata_cols = [col for col in metadata_cols if col in final_recs.columns]
    if len(valid_metadata_cols) != len(metadata_cols):
        print(f"Warning: Not all requested metadata columns ({metadata_cols}) found in results.")

    if final_recs.empty:
        print("No recommendations found after filtering.")
        return pd.DataFrame(columns=valid_metadata_cols)

    return final_recs[valid_metadata_cols] # Return as DataFrame


# --- Usage Example ---
print("\n--- Generating Recommendations ---")
# Define the input songs
my_playlist = [{'name': 'Come As You Are', 'year': 1991},
               {'name': 'Smells Like Teen Spirit', 'year': 1991},
               {'name': 'Lithium', 'year': 1992},
               {'name': 'All Apologies', 'year': 1993},
               {'name': 'Stay Away', 'year': 1993},
               {'name': 'NonExistent Song', 'year': 2000}] # Example of a song not in the dataset

# Ensure the pipeline exists before calling
if 'song_cluster_pipeline' in locals() and song_cluster_pipeline is not None:
    recommendations = recommend_songs(
        song_list=my_playlist,
        spotify_data=data, # Pass the main DataFrame
        pipeline=song_cluster_pipeline, # Pass the *trained* pipeline
        feature_cols=number_cols, # Pass the *consistent* list of numeric columns
        n_songs=10
    )

    print("\nRecommended Songs:")
    if not recommendations.empty:
        print(recommendations.to_string())
    else:
        print("Could not generate recommendations.")
else:
    print("Skipping recommendation generation because the pipeline was not successfully trained.")


# --- Save the Pipeline ---
# Only save if it was trained successfully
if 'song_cluster_pipeline' in locals() and song_cluster_pipeline is not None:
    pipeline_filename = "music_recommendation_pipeline.pkl" # Changed name slightly
    try:
        joblib.dump(song_cluster_pipeline, pipeline_filename)
        print(f"\nPipeline successfully saved to {pipeline_filename}")
    except Exception as e:
        print(f"\nError saving pipeline: {e}")

Error loading data: read_csv() got an unexpected keyword argument 'errors'

--- Data Info ---
data.csv info:


NameError: name 'data' is not defined

: 